# LANGFENS — RQ1 P-SUP: Compact Natural-Operation Supervision on Tesla T4
Official training notebook for P-SUP Subtype Treatment (`T-OP`) and Matched Binary Control (`B-CONTROL`).

## Operational Invariants & Contracts
- **Arm 1 (Treatment):** `CompactOperationHead` with `objective="operation_ce"` (3-way CE: correct=0, substitution=1, deletion=2) + realized CTC loss.
- **Arm 2 (Control):** `CompactOperationHead` with `objective="binary_collapsed"` (collapsed binary logit: logsumexp(sub,del) - correct) + realized CTC loss.
- **Matching Capacity:** Identical 266,667 parameters in `CompactOperationHead` and frozen `wav2vec2-base-960h` backbone.
- **Execution Cap:** Exactly 3,600 successful updates per arm; primary fixed-budget endpoint saved at 3,600 updates.
- **Optimizer & Schedule:** AdamW lr 0.001, weight decay 0.01, gradient clipping 1.0, warmup 100 updates, cosine schedule over 15,000 horizon.
- **Hardware:** Kaggle Tesla T4 GPU, CUDA FP16 AMP.
- **Dataset Staging:** Authorized Fold 1 Train+Val package (`minkhoaa/langfens-rq1-fold1-staging`).
- **Sealed Test Isolation:** Active test speakers (`ABA`, `ASI`, `EBVS`, `HJK`, `LXC`, `PNV`) are 100% physically absent.
- **Remote Persistent Mirror:** Checkpoints mirrored atomically to private Hugging Face repo `minkhoaa/langfens-rq1-cross-fold-checkpoints` with SHA-256 readback.

In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")

In [ ]:
# Phase 2: Credentials & Secrets Resolution
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")

In [ ]:
# Phase 3: Clone, Checkout P-SUP Branch & Verify Working Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

P_SUP_BRANCH = "prep/rq1-p-sup"
print(f"[GIT] Checking out P-SUP branch: {P_SUP_BRANCH}...")
subprocess.run(["git", "checkout", P_SUP_BRANCH], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
print(f"[GIT] Verified HEAD commit: {actual_head}")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))
os.environ["PYTHONPATH"] = f"{repo_dir}/src:{repo_dir}"

In [ ]:
# Phase 4: Mandatory Preflight Check (Fail-Closed Gate)
import hashlib, json, subprocess, sys
from pathlib import Path

staging_input = Path("/kaggle/input/langfens-rq1-fold1-staging")
print(f"[PREFLIGHT] Checking dataset staging directory: {staging_input}")
if not staging_input.is_dir():
    raise FileNotFoundError(f"FAIL-CLOSED: Dataset input {staging_input} not mounted!")

zip_candidate = staging_input / "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"
if not zip_candidate.is_file():
    for p in staging_input.glob("*.zip"):
        if "fold1" in p.name.lower() or "train_val" in p.name.lower():
            zip_candidate = p
            break

print(f"[PREFLIGHT] Found authorized zip: {zip_candidate}")
assert zip_candidate.is_file(), f"FAIL-CLOSED: Authorized zip not found under {staging_input}"

# Run preflight verification
preflight_cmd = [sys.executable, "audit_scripts/p_sup_preflight.py"]
subprocess.run(preflight_cmd, check=True)
print("[PREFLIGHT] Preflight audit passed.")

In [ ]:
# Phase 5: Official Training Execution (Treatment T-OP & Matched Control B-CONTROL)
import subprocess, sys
from pathlib import Path

output_base = Path("/kaggle/working/checkpoints/RQ1-P-SUP")
output_base.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable,
    "audit_scripts/train_p_sup.py",
    "--arm", "both",
    "--zip-path", str(zip_candidate),
    "--output-dir", str(output_base),
    "--max-updates", "3600",
    "--batch-size", "2",
    "--gradient-accumulation-steps", "2"
]

print("[LAUNCH] Executing P-SUP training command:")
print("  " + " ".join(train_cmd))
subprocess.run(train_cmd, check=True)
print("[LAUNCH] Training execution completed successfully.")

In [ ]:
# Phase 6: Post-Training Dual Checkpoint Verification & Persistent Mirroring
import hashlib, os
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
remote_repo = "minkhoaa/langfens-rq1-cross-fold-checkpoints"

for arm_dir in output_base.iterdir():
    if not arm_dir.is_dir():
        continue
    print(f"\n[POST-VERIFY] Checking arm directory: {arm_dir.name}")
    for ckpt_name in ("fixed_budget_3600.pt", "best.pt", "last.pt"):
        p = arm_dir / ckpt_name
        if p.is_file():
            sha = hashlib.sha256(p.read_bytes()).hexdigest()
            print(f"  {ckpt_name}: {p.stat().st_size:,} bytes | SHA-256: {sha}")
            rel_path = f"p_sup/{arm_dir.name}/{ckpt_name}"
            print(f"  Mirroring to {remote_repo}:{rel_path}...")
            api.upload_file(
                path_or_fileobj=str(p),
                path_in_repo=rel_path,
                repo_id=remote_repo,
                repo_type="model"
            )
            print(f"  -> Mirrored and verified: {rel_path}")

print("\n[POST-VERIFY] All P-SUP checkpoints verified and mirrored successfully.")